# Spectral Shape Features (IEMOCAP)

This notebook extracts spectral-shape contours (centroid, rolloff, contrast, etc.)
and summarizes them per utterance.

In [1]:
from pathlib import Path

import librosa
import numpy as np
import pandas as pd


In [2]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "spectral_shape"
OUT_FILE = "spectral_shape_features.csv"

# Audio + feature params
TARGET_SR = 16_000
FRAME_LENGTH = 2048
HOP_LENGTH = 512
ROLL_PERCENT = 0.85

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


In [3]:
def load_audio(path: Path) -> tuple[np.ndarray, int]:
    # Load audio and resample to TARGET_SR so features are comparable
    audio, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return audio, sr


def _compute_flux(magnitude: np.ndarray) -> np.ndarray:
    # Compute spectral flux from a magnitude spectrogram
    if magnitude.shape[1] <= 1:
        return np.full(magnitude.shape[1], np.nan)
    diff = np.diff(magnitude, axis=1)
    flux = np.sqrt(np.sum(diff ** 2, axis=0))
    return np.pad(flux, (1, 0), mode="constant", constant_values=np.nan)


def compute_spectral_shape(audio: np.ndarray, sr: int) -> dict[str, np.ndarray]:
    # Compute spectral-shape contours
    stft = librosa.stft(audio, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)
    magnitude = np.abs(stft)

    centroid = librosa.feature.spectral_centroid(S=magnitude, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=magnitude, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(
        S=magnitude,
        sr=sr,
        roll_percent=ROLL_PERCENT,
    )[0]
    flatness = librosa.feature.spectral_flatness(S=magnitude)[0]
    flux = _compute_flux(magnitude)
    contrast = librosa.feature.spectral_contrast(S=magnitude, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(
        audio,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH,
    )[0]

    return {
        "centroid": centroid,
        "bandwidth": bandwidth,
        "rolloff": rolloff,
        "flatness": flatness,
        "flux": flux,
        "contrast": contrast,
        "zcr": zcr,
    }


def summarize_curve(prefix: str, values: np.ndarray) -> dict[str, float]:
    # Summary stats ignoring NaNs
    vec = np.asarray(values, dtype=float).ravel()
    vec = vec[~np.isnan(vec)]
    if vec.size == 0:
        return {
            f"{prefix}_mean": float("nan"),
            f"{prefix}_std": float("nan"),
            f"{prefix}_min": float("nan"),
            f"{prefix}_max": float("nan"),
            f"{prefix}_median": float("nan"),
        }
    return {
        f"{prefix}_mean": float(vec.mean()),
        f"{prefix}_std": float(vec.std()),
        f"{prefix}_min": float(vec.min()),
        f"{prefix}_max": float(vec.max()),
        f"{prefix}_median": float(np.median(vec)),
    }


def extract_spectral_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    result = compute_spectral_shape(audio, sr)
    features: dict[str, float] = {}
    features.update(summarize_curve("spec_centroid", result["centroid"]))
    features.update(summarize_curve("spec_bandwidth", result["bandwidth"]))
    features.update(summarize_curve("spec_rolloff", result["rolloff"]))
    features.update(summarize_curve("spec_flatness", result["flatness"]))
    features.update(summarize_curve("spec_flux", result["flux"]))
    features.update(summarize_curve("zcr", result["zcr"]))

    for idx, band in enumerate(result["contrast"]):
        features.update(summarize_curve(f"spec_contrast_b{idx}", band))
    return features


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: valid emotion + agreement > 0
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape


In [5]:
rows: list[dict[str, float | str | int]] = []
missing: list[str] = []

for _, row in df.iterrows():
    rel_path = row["path"]
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        missing.append(str(audio_path))
        continue

    audio, sr = load_audio(audio_path)
    duration_s = audio.shape[0] / sr
    features = extract_spectral_features(audio, sr)

    record: dict[str, float | str | int] = {
        "path": str(rel_path),
        "session": int(row["session"]),
        "method": row["method"],
        "gender": row["gender"],
        "emotion": row["emotion"],
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    rows.append(record)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape
